In [125]:
#get dataset
#use weights and bias to score them (Wx+b)
#use softmax (any other function) to get probabilities
#calculate loss
#optimize
#repeat until fine

#what we haven't done
#model validation

In [126]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/iris.data", header=None)
#can do data preprocessing like mean subtraction and standardization
df.head()

,0,1,2,3,4
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


In [127]:
df = np.array(df)

x = df[:, :-1].astype(np.float64)
classes, y = np.unique(df[:, -1], return_inverse=True)

indices = np.random.permutation(len(x))
x_shuffled = x[indices]
y_shuffled = y[indices]

#split the data
x_train = x_shuffled[:100]
x_test = x_shuffled[100:]

y_train = y_shuffled[:100]
y_test = y_shuffled[100:]

In [128]:
input = x_train

# initialize weights (3 classes × 4 features)
h1 = np.random.randn(4, 32) * 0.01
h2 = np.random.randn(32, 3) * 0.01

# initialize bias (3 classes)
b1 = np.zeros((1, 32))
b2 = np.zeros((1, 3))

In [129]:
EPOCH = 1000
LAMBDA = 0.0001
LEARNING_RATE = 0.01

Y = np.eye(3)[y_train]

for epoch in range(EPOCH):

    # forward propagation
    hidden_pre_activation = input @ h1 + b1
    hidden_output = np.maximum(0, hidden_pre_activation)
    scores = hidden_output @ h2 + b2


    # softmax
    exponent_scores = np.exp(scores - np.max(scores, axis=1, keepdims=True))
    exponentsum = np.sum(exponent_scores, axis=1, keepdims=True)
    probabilities = exponent_scores / exponentsum


    # loss
    true_probs = probabilities[np.arange(len(y_train)), y_train]
    loss = -np.mean(np.log(true_probs + 1e-12)) + LAMBDA * (np.sum(h1**2) + np.sum(h2**2))


    # backpropagation
    dl_ds = (probabilities - Y) / len(y_train)

    # output layer gradients
    dl_dh2 = hidden_output.T @ dl_ds + 2 * LAMBDA * h2
    dl_db2 = np.sum(dl_ds, axis=0, keepdims=True)

    # propagate backwards through second layer
    dl_dhidden_output = dl_ds @ h2.T

    # differentiate through ReLU
    dl_dhidden_pre_activation = dl_dhidden_output * (hidden_pre_activation > 0)

    # first layer gradients
    dl_dh1 = input.T @ dl_dhidden_pre_activation + 2 * LAMBDA * h1
    dl_db1 = np.sum(dl_dhidden_pre_activation, axis=0, keepdims=True)


    # gradient descent
    h1 -= LEARNING_RATE * dl_dh1
    b1 -= LEARNING_RATE * dl_db1

    h2 -= LEARNING_RATE * dl_dh2
    b2 -= LEARNING_RATE * dl_db2


    if epoch % 100 == 0:
        print(f"epoch {epoch}, loss: {loss}")

epoch 0, loss: 1.0986634741749752
epoch 100, loss: 1.0845611348612734
epoch 200, loss: 1.0388281084766324
epoch 300, loss: 0.9563052334278005
epoch 400, loss: 0.8236116388004561
epoch 500, loss: 0.6867634086808397
epoch 600, loss: 0.5732299602461971
epoch 700, loss: 0.49966484641682857
epoch 800, loss: 0.4453105729500767
epoch 900, loss: 0.39880227149482667


In [130]:
# forward propagation
hidden_pre_activation = x_test @ h1 + b1
hidden_output = np.maximum(0, hidden_pre_activation)
scores = hidden_output @ h2 + b2


# softmax
exponent_scores = np.exp(scores - np.max(scores, axis=1, keepdims=True))
exponentsum = np.sum(exponent_scores, axis=1, keepdims=True)
probabilities = exponent_scores / exponentsum


In [131]:
correct = 0
for i, j in enumerate(probabilities):
  if np.argmax(j) == y_test[i]:
    correct += 1

print("Accuracy: " + str(correct/len(y_test)))

Accuracy: 0.98
